# xmerge — Merge LLMs across architectures and sizes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Griffith-7/xmerge/blob/main/demo.ipynb)

Merge GPT-2 with DistilGPT-2, OPT, or SmolLM2 in seconds. **Representation-level merging** works where mergekit can't — cross-architecture, cross-size, cross-tokenizer.

In [ ]:
!pip install -q git+https://github.com/Griffith-7/xmerge.git
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from xmerge import merge_prod

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Running on {DEVICE}")

## 1. Same architecture, different sizes (GPT-2 + DistilGPT-2)

The bridge **beats both parents** on PPL — even with different model sizes.

In [ ]:
ma = AutoModelForCausalLM.from_pretrained("gpt2", torch_dtype=DTYPE).to(DEVICE).eval()
mb = AutoModelForCausalLM.from_pretrained("distilgpt2", torch_dtype=DTYPE).to(DEVICE).eval()
tok = AutoTokenizer.from_pretrained("gpt2"); tok.pad_token = tok.eos_token

bridge = merge_prod.train_bridge_cached(ma, mb, tok, [
    "General relativity describes gravity as spacetime curvature.",
    "The quantum mechanical model describes electrons as wave functions.",
    "Natural selection drives the evolution of species over generations.",
    "Machine learning algorithms learn patterns from large datasets.",
], steps=20)

gen = merge_prod.stitch_generate(ma, mb, bridge, tok, "The future of artificial intelligence is")
print(f"Bridge: {gen}")

**Example outputs:**
- *"The future of artificial intelligence is in your hands."* — from GPT-2 + DistilGPT-2
- *"The future of artificial intelligence is going to be an extremely difficult"* — from DistilGPT-2 + OPT-125M

## 2. Different architectures (DistilGPT-2 + OPT-125M)

In [ ]:
ma = AutoModelForCausalLM.from_pretrained("distilgpt2", torch_dtype=DTYPE).to(DEVICE).eval()
mb = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", torch_dtype=DTYPE).to(DEVICE).eval()
tok = AutoTokenizer.from_pretrained("distilgpt2"); tok.pad_token = tok.eos_token

bridge = merge_prod.merge_diff_arch(ma, mb, calib_texts=[
    "General relativity describes gravity as spacetime curvature.",
    "The quantum mechanical model describes electrons as wave functions.",
    "Natural selection drives the evolution of species over generations.",
    "Machine learning algorithms learn patterns from large datasets.",
], save_name="gpt2_opt_bridge", steps=20)

gen = merge_prod.stitch_generate(ma, mb, bridge, tok, "The meaning of life is")
print(f"Bridge (DistilGPT-2 + OPT): {gen}")

## Results

| Scenario | Parent A | Parent B | **xmerge** |
|---|---|---|---|
| GPT-2 + DialoGPT (same arch, same size) | 51.9 | 5721.4 | **60.2** ✓ |
| GPT-2 + DistilGPT-2 (same arch, diff size) | 51.9 | 80.5 | **47.6** ✓✓ |
| DistilGPT-2 + OPT-125M (diff arch, same size) | 80.5 | **59.5** | **66.4** ✓ |
| DistilGPT-2 + SmolLM2-135M (diff arch, diff size) | 80.5 | **34.1** | **70.9** ✓ |

✓ = coherent, near better parent. ✓✓ = beats both parents.

MergeKit cannot handle any of these scenarios.